In [10]:
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from tavily import TavilyClient
from langchain.agents import Tool, initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.memory import ConversationBufferMemory

# 🔑 GROQ & tavily API Key
GROQ_API_KEY = ""
tavily_api_key = ""

# 🔸 Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=300,
)

# Simple QA tool (LLMChain)
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")
qa_chain =  qa_prompt|llm
qa_tool = Tool(
    name="Simple_QA",
    func=qa_chain.invoke,
    description="Answer the question without hallucination"
)

# Web search tool (Tavily)
tavily = TavilyClient(api_key=tavily_api_key)
search_tool = Tool(
    name="Web_Search",
    func=tavily.search,
    description="Search the internet for current info"
)

tools = [qa_tool, search_tool]
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [12]:
#1) Zero-Shot ReAct (simple + reliable default)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)
result = agent.invoke("Summarize LangChain in 2 lines, then tell me who created it.")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: To answer this question, I need to first understand what LangChain is and who created it. Since I don't have any prior knowledge about LangChain, I should start by searching for information about it on the internet.

Action: Web_Search
Action Input: query="LangChain summary" + " LangChain creator", search_depth="basic"
Observation: {'query': 'query=LangChain summary +  LangChain creator, search_depth=basic', 'response_time': 1.14, 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.youtube.com/watch?v=p_MQRWH5Y6k', 'title': 'Summarizing and Querying Multiple Papers with LangChain', 'content': "In this video, I'll show you how to summarize and query multiple research papers using the LangChain library. We'll be working with three", 'score': 0.99448806, 'raw_content': None}, {'url': 'https://reference.langchain.com/python/langchain-classic/memory/summary', 'title': 'summary | langchain_classic - LangChain Reference Docs', 'content': 'Chain

In [13]:
#2) Conversational ReAct (with memory; non-chat model baseline)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: Do I need to use a tool? No
AI: LangChain is a framework for building applications that utilize large language models, allowing developers to create more complex and interactive interfaces. It was created by Harrison Chase and other contributors, with the goal of simplifying the process of integrating language models into various applications.

> Finished chain.


In [14]:
#3) Chat Conversational ReAct (recommended for GPT-3.5/4 chat)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Plan a 3-step study path for LangChain.")
#result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


```json
{
    "action": "Simple_QA",
    "action_input": "LangChain study path"
}
```
Observation: content="LangChain is a framework for building applications that utilize large language models. Here's a suggested study path to learn LangChain:\n\n**Step 1: Prerequisites**\n\n1. **Python programming**: Familiarize yourself with Python basics, such as data structures, functions, and object-oriented programming.\n2. **Large Language Models (LLMs)**: Understand the fundamentals of LLMs, including their architecture, training, and applications.\n3. **Natural Language Processing (NLP)**: Learn the basics of NLP, including text preprocessing, tokenization, and sentiment analysis.\n\n**Step 2: LangChain Fundamentals**\n\n1. **Introduction to LangChain**: Learn about the LangChain framework, its goals, and its components.\n2. **LangChain Architecture**: Study the architecture of LangChain, including the role of LLMs, agents, and applications.\n3. **LangChain Core Concepts**: Understand key con

In [15]:
from langchain.tools import StructuredTool
from langchain.agents import initialize_agent, AgentType
from pydantic import BaseModel

# ✅ Define the input schema using Pydantic
class TitleInput(BaseModel):
    topic: str
    tone: str = "concise"

# ✅ Define the tool function
def title_tool_fn(topic: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {topic} in Practice"

# ✅ Create the structured tool
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)

# ✅ Initialize the agent
agent = initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ✅ Run the agent
result = agent.run("Make a friendly title about LangGraph tutorials.")
print("✅ Result:", result)


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Action:
```
{
  "action": "TitleMaker",
  "action_input": {
    "topic": "LangGraph tutorials",
    "tone": "friendly"
  }
}
```

Observation: Friendly Title: LangGraph tutorials in Practice
Thought:Action:
```
{
  "action": "Final Answer",
  "action_input": "LangGraph tutorials in Practice"
}
```

> Finished chain.
✅ Result: LangGraph tutorials in Practice


In [18]:
#5)OpenAI Functions (function-calling schema)
print(llm)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)
result = agent.run("Search the web for LangGraph docs and give me 3 bullets.")


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


client=<groq.resources.chat.completions.Completions object at 0x0000023FD07A7040> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023FD07A4700> model_name='llama-3.3-70b-versatile' temperature=1e-08 groq_api_key=SecretStr('**********') max_tokens=300


> Finished chain.


In [19]:
#6) OpenAI Multi-Functions (can call multiple tools in one step) groq cant support gpt 
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_MULTI_FUNCTIONS,
    verbose=True
)
result = agent.run("Answer briefly from QA; also check the web for any updates.")


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")




> Finished chain.


In [23]:
#7) Self-Ask with Search (clarifies, then searches)
tavily = TavilyClient(api_key=tavily_api_key)
search_tool = Tool(
    name="Intermediate Answer",  # This name is REQUIRED
    func=tavily.search,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)
result = agent.run("When was the first LangChain release and who founded it?")


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Could not parse output: To answer this question, follow-up questions are needed. 

Follow up: What is LangChain?
(This question is necessary to understand what LangChain is and its relevance.)

Intermediate answer: Invalid or incomplete response
To answer the question about LangChain, follow-up questions are indeed necessary. 

Follow up: What is LangChain?
Intermediate answer: {'query': 'What is LangChain?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://aws.amazon.com/what-is/langchain/', 'title': 'What is LangChain? - AWS', 'content': 'LangChain is an open source framework for building applications based on large language models (LLMs). LLMs are large deep-learning models pre-trained on large', 'score': 0.94451064, 'raw_content': None}, {'url': 'https://www.langchain.com/langchain', 'title': 'Open Source AI Agent Framework | Build Agents Faster - LangChain', 'content': "# Build agents faster, your way. LangChain is an open source framework wi

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.3-70b-versatile` in organization `org_01kpq8hn5vfsavs1v8k5h5xsvc` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 12074, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

![Model Diagram](agenttypes1.png)

![Model Diagram](agenttype2.png)

In [ ]:
Deep Dive

![Model Diagram](pro1.png)

![Model Diagram](pro2.png)

![Model Diagram](pro3.png)